# Unified Training & Testing Pipeline (Model 1 & Model 2)

This notebook provides a complete pipeline for:
1.  **Setup**: Mounting Google Drive and installing dependencies.
2.  **Data Preparation**: Extracting the UDC-SIT dataset from Drive to local disk for fast training.
3.  **Training**: Training the Teacher and Student models (supporting both `v1` and `v2` variants).
4.  **Testing**: Evaluating the trained models on the **Testing** split and saving results to Drive.

### Configuration
Adjust the parameters in the first code cell below.

In [ ]:
# --- CONFIGURATION ---

# Model Selection
MODEL_VARIANT = "v2"  # Options: "v1" (Amplitude Only), "v2" (Amplitude + Phase + MultiScale)

# Data Settings
USE_SUBSET = False    # If True, trains on a small subset (useful for debugging)
FORCE_EXTRACT = False # If True, re-extracts dataset even if it exists

# Paths (Google Drive)
DRIVE_ROOT = "/content/drive/MyDrive/Computational Imaging Project"
DATASET_TAR_DIR = f"{DRIVE_ROOT}/Datasets/UDC-SIT"  # Folder containing .tar files
DATASET_EXTRACTED_DRIVE = f"{DRIVE_ROOT}/Datasets/UDC-SIT/UDC-SIT" # Folder containing extracted 'training', 'validation', 'testing'
RESULTS_DIR = f"{DRIVE_ROOT}/Results/Model_{MODEL_VARIANT.upper()}"

# Training Params
TEACHER_EPOCHS = 22
STUDENT_EPOCHS = 50
BATCH_SIZE = 4

if USE_SUBSET:
    TEACHER_EPOCHS = 2
    STUDENT_EPOCHS = 2
    RESULTS_DIR += "_Subset"

print(f"Configuration:\nVariant: {MODEL_VARIANT}\nSubset: {USE_SUBSET}\nResults: {RESULTS_DIR}")

## 1. Environment Setup

In [ ]:
from google.colab import drive
import os
import sys
import glob

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Clone Repo (if not already present)
%cd /content
if not os.path.exists("CSC2529-Fall-Project"):
    !git clone https://github.com/navdeepsingh1609/CSC2529-Fall-Project.git

%cd CSC2529-Fall-Project

# Install Dependencies
!pip install -q -r requirements.txt
!pip install -q timm mamba-ssm causal-conv1d

# Initialize Submodules (MambaIR)
!git submodule update --init --recursive --remote

## 2. Data Preparation
We extract the dataset to `/content/dataset` for faster I/O during training.

In [ ]:
import shutil
import tarfile

# We will determine the actual root dynamically after extraction/linking
BASE_EXTRACT_PATH = "/content/dataset/UDC-SIT"
os.makedirs(BASE_EXTRACT_PATH, exist_ok=True)

def prepare_data():
    # Check if data already exists locally (heuristic: check for 'training' folder)
    # We search for 'training' inside BASE_EXTRACT_PATH
    existing_train = glob.glob(f"{BASE_EXTRACT_PATH}/**/training", recursive=True)
    if existing_train and not FORCE_EXTRACT:
        print(f"Data found locally at {os.path.dirname(existing_train[0])}. Skipping extraction.")
        return os.path.dirname(existing_train[0])

    print("Preparing dataset...")
    
    # Option A: Extract from Tar (Preferred for speed)
    tar_files = [f for f in os.listdir(DATASET_TAR_DIR) if f.endswith('.tar')]
    
    if tar_files:
        print(f"Found tar files: {tar_files}. Extracting...")
        for tar_name in tar_files:
            tar_path = os.path.join(DATASET_TAR_DIR, tar_name)
            print(f"Extracting {tar_name}...")
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=BASE_EXTRACT_PATH)
    else:
        # Option B: Copy/Link from Extracted Folder on Drive
        print("No tar files found. Linking/Copying from extracted Drive folder...")
        
        # We'll create a structure that matches what we expect: BASE_EXTRACT_PATH/UDC-SIT/...
        target_root = os.path.join(BASE_EXTRACT_PATH, "UDC-SIT")
        os.makedirs(target_root, exist_ok=True)
        
        # Link Training (Copying is better for speed but slow to setup, using symlink for now if copy fails or is too slow)
        # Actually, for Colab training speed, copying 'training' is highly recommended.
        if not os.path.exists(f"{target_root}/training"):
             print("Copying training data (this may take a while)...")
             try:
                 shutil.copytree(f"{DATASET_EXTRACTED_DRIVE}/training", f"{target_root}/training")
             except Exception as e:
                 print(f"Copy failed ({e}), falling back to symlink...")
                 os.symlink(f"{DATASET_EXTRACTED_DRIVE}/training", f"{target_root}/training")
        
        # Link Validation
        if not os.path.exists(f"{target_root}/validation"):
             os.symlink(f"{DATASET_EXTRACTED_DRIVE}/validation", f"{target_root}/validation")
             
        # Link Testing
        if not os.path.exists(f"{target_root}/testing"):
             os.symlink(f"{DATASET_EXTRACTED_DRIVE}/testing", f"{target_root}/testing")

    # Find the root containing 'training'
    found_train = glob.glob(f"{BASE_EXTRACT_PATH}/**/training", recursive=True)
    if not found_train:
        raise RuntimeError("Could not find 'training' folder after extraction/linking!")
    
    final_root = os.path.dirname(found_train[0])
    print(f"Data preparation complete. Root: {final_root}")
    return final_root

LOCAL_DATA_ROOT = prepare_data()

## 3. Training: Teacher Model
We train the teacher first. Checkpoints are saved to Drive.

In [ ]:
# Construct arguments
teacher_args = [
    f"--model-variant {MODEL_VARIANT}",
    f"--data-root '{LOCAL_DATA_ROOT}'",
    f"--num-epochs {TEACHER_EPOCHS}",
    f"--batch-size {BATCH_SIZE}",
    f"--drive-checkpoint-dir '{RESULTS_DIR}/checkpoints'",
    "--save-loss-history"
]

if USE_SUBSET:
    teacher_args.append("--max-train-images 50")
    teacher_args.append("--max-val-images 10")

cmd = f"python train_teacher.py {' '.join(teacher_args)}"
print(f"Running: {cmd}")
!{cmd}

## 4. Training: Student Model (Knowledge Distillation)
Now we train the student using the trained teacher.

In [ ]:
# Find the best teacher checkpoint
import glob
ckpt_dir = f"{RESULTS_DIR}/checkpoints"
teacher_ckpts = sorted(glob.glob(f"{ckpt_dir}/teacher_*.pth"))
if not teacher_ckpts:
    raise FileNotFoundError("No teacher checkpoint found! Did training fail?")
best_teacher = teacher_ckpts[-1] # Assume last is best or latest
print(f"Using teacher checkpoint: {best_teacher}")

student_args = [
    f"--model-variant {MODEL_VARIANT}",
    f"--data-root '{LOCAL_DATA_ROOT}'",
    f"--teacher-weights '{best_teacher}'",
    f"--num-epochs {STUDENT_EPOCHS}",
    f"--batch-size {BATCH_SIZE}",
    f"--drive-checkpoint-dir '{RESULTS_DIR}/checkpoints'",
    "--save-loss-history"
]

if USE_SUBSET:
    student_args.append("--max-train-images 50")
    student_args.append("--max-val-images 10")

cmd = f"python train_student_kd.py {' '.join(student_args)}"
print(f"Running: {cmd}")
!{cmd}

## 5. Testing & Evaluation
Evaluate both models on the **Testing** split. Results (metrics, plots, .npy files) are saved to Drive.

In [ ]:
# Find checkpoints
student_ckpts = sorted(glob.glob(f"{ckpt_dir}/student_*.pth"))
best_student = student_ckpts[-1] if student_ckpts else None

if best_student:
    print(f"Evaluating Student: {best_student}")
    test_args = [
        "--model-type student",
        f"--model-variant {MODEL_VARIANT}",
        f"--data-root '{LOCAL_DATA_ROOT}'",
        f"--checkpoint-path '{best_student}'",
        f"--results-root 'results_temp_student'",
        f"--drive-results-root '{RESULTS_DIR}/evaluation_student'",
        "--save-npy"
    ]
    cmd = f"python testing_udc.py {' '.join(test_args)}"
    print(f"Running: {cmd}")
    !{cmd}
else:
    print("No student checkpoint found to evaluate.")

if best_teacher:
    print(f"Evaluating Teacher: {best_teacher}")
    test_args = [
        "--model-type teacher",
        f"--model-variant {MODEL_VARIANT}",
        f"--data-root '{LOCAL_DATA_ROOT}'",
        f"--checkpoint-path '{best_teacher}'",
        f"--results-root 'results_temp_teacher'",
        f"--drive-results-root '{RESULTS_DIR}/evaluation_teacher'",
        "--save-npy"
    ]
    cmd = f"python testing_udc.py {' '.join(test_args)}"
    print(f"Running: {cmd}")
    !{cmd}